# MLP Circuit Analysis

Analyze MLP layers as circuits: trace neuron-to-logit paths, classify neurons as
knowledge vs feature detectors, decompose contributions, and compare MLP layers.

## Why This Matters

MLP circuit provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_circuit_analysis import (
    neuron_to_logit_paths,
    mlp_knowledge_vs_feature,
    mlp_contribution_decomposition,
    mlp_nonlinearity_effect,
    mlp_layer_comparison,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Neuron-to-Logit Paths

Trace how individual neurons influence output logits via W_out @ W_U.

In [ ]:
result = neuron_to_logit_paths(model, tokens, layer=0, top_k=5)
print(f'Layer 0 — top {len(result["neurons"])} neurons by logit influence:\n')
for n in result['neurons']:
    print(f"  Neuron {n['neuron_idx']:3d}: total_abs_logit={n['total_abs_logit']:.4f}")
    for tok in n['top_promoted'][:3]:
        print(f"    promotes token {tok['token']:3d} with logit {tok['logit_contribution']:+.4f}")

## Knowledge vs Feature Neurons

Classify neurons: *knowledge neurons* have sparse activation + concentrated logit effect;
*feature neurons* activate broadly.

In [ ]:
result = mlp_knowledge_vs_feature(model, tokens, layer=0)
print(f"Knowledge neurons: {result['n_knowledge']}")
print(f"Feature neurons:   {result['n_feature']}")
print(f"Knowledge fraction: {result['knowledge_fraction']:.2%}")

## Contribution Decomposition

Decompose the MLP output into per-neuron contributions and measure concentration (Gini).

In [ ]:
result = mlp_contribution_decomposition(model, tokens, layer=0)
print(f"Total contribution: {result['total_contribution']:.4f}")
print(f"Active neurons: {result['n_neurons']}")
print(f"Gini coefficient: {result['gini']:.4f}")
print(f'\nTop contributing neurons:')
for n in result['top_neurons'][:5]:
    print(f"  Neuron {n['neuron_idx']:3d}: contribution={n['contribution']:.4f} ({n['fraction']:.1%} of total)")

## Nonlinearity Effect

Compare pre- and post-activation representations to understand the activation function's role.

In [ ]:
result = mlp_nonlinearity_effect(model, tokens, layer=0)
print(f"Pre-activation norm:  {result['pre_norm']:.4f}")
print(f"Post-activation norm: {result['post_norm']:.4f}")
print(f"Pre→Post cosine sim:  {result['pre_post_cosine']:.4f}")
print(f"Pre near-zero frac:   {result['pre_near_zero']:.2%}")
print(f"Post near-zero frac:  {result['post_near_zero']:.2%}")

## MLP Layer Comparison

Compare MLP behavior across all layers.

In [ ]:
result = mlp_layer_comparison(model, tokens)
print(f"Most active MLP layer: {result['most_active_layer']}\n")
for layer in result['per_layer']:
    print(f"Layer {layer['layer']}: output_norm={layer['output_norm']:.4f}, "
          f"sparsity={layer['activation_sparsity']:.2%}, "
          f"mean_act={layer['mean_activation']:.4f}")